# Аналіз даних у PySpark

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('goit-de-hw-03').getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/19 20:02:55 WARN Utils: Your hostname, ROG, resolves to a loopback address: 127.0.1.1; using 172.19.41.224 instead (on interface eth0)
26/02/19 20:02:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/19 20:02:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Директорія з даними
data_dir = Path('../data')

users_path = str(data_dir / 'users.csv')
purchases_path = str(data_dir / 'purchases.csv')
products_path = str(data_dir / 'products.csv')

In [4]:
# 1. Завантаження CSV у DataFrame
users = spark.read.csv(users_path, header=True, inferSchema=True)
purchases = spark.read.csv(purchases_path, header=True, inferSchema=True)
products = spark.read.csv(products_path, header=True, inferSchema=True)

users.show(5, truncate=False)
purchases.show(5, truncate=False)
products.show(5, truncate=False)

+-------+------+---+-----------------+
|user_id|name  |age|email            |
+-------+------+---+-----------------+
|1      |User_1|45 |user1@example.com|
|2      |User_2|48 |user2@example.com|
|3      |User_3|36 |user3@example.com|
|4      |User_4|46 |user4@example.com|
|5      |User_5|29 |user5@example.com|
+-------+------+---+-----------------+
only showing top 5 rows
+-----------+-------+----------+----------+--------+
|purchase_id|user_id|product_id|date      |quantity|
+-----------+-------+----------+----------+--------+
|1          |52     |9         |2022-01-01|1       |
|2          |93     |37        |2022-01-02|8       |
|3          |15     |33        |2022-01-03|1       |
|4          |72     |42        |2022-01-04|9       |
|5          |61     |44        |2022-01-05|6       |
+-----------+-------+----------+----------+--------+
only showing top 5 rows
+----------+------------+-----------+-----+
|product_id|product_name|category   |price|
+----------+------------+-----------

In [5]:
# 2. Очищення даних (видалення пропусків) + тимчасові SQL-view
users.dropna().createOrReplaceTempView('users')
purchases.dropna().createOrReplaceTempView('purchases')
products.dropna().createOrReplaceTempView('products')

In [6]:
# 3. Загальна сума покупок за категоріями
total_by_category = spark.sql("""
SELECT
    pr.category,
    ROUND(SUM(p.quantity * pr.price), 2) AS total_spent
FROM purchases p
JOIN products pr ON p.product_id = pr.product_id
GROUP BY pr.category
ORDER BY total_spent DESC
""")

total_by_category.show(truncate=False)

+-----------+-----------+
|category   |total_spent|
+-----------+-----------+
|Sports     |1802.5     |
|Home       |1523.5     |
|Electronics|1174.8     |
|Clothing   |790.3      |
|Beauty     |459.9      |
+-----------+-----------+



In [7]:
# 4. Сума покупок за категоріями для віку 18-25
spent_18_25_by_category = spark.sql("""
SELECT
    pr.category,
    ROUND(SUM(p.quantity * pr.price), 2) AS spent_18_25
FROM purchases p
JOIN products pr ON p.product_id = pr.product_id
JOIN users u ON p.user_id = u.user_id
WHERE u.age BETWEEN 18 AND 25
GROUP BY pr.category
ORDER BY spent_18_25 DESC
""")

spent_18_25_by_category.show(truncate=False)

+-----------+-----------+
|category   |spent_18_25|
+-----------+-----------+
|Home       |361.1      |
|Sports     |310.5      |
|Electronics|249.6      |
|Clothing   |245.0      |
|Beauty     |41.4       |
+-----------+-----------+



In [8]:
# 5. Частка витрат за категоріями (%) для 18-25
share_18_25_by_category = spark.sql("""
WITH category_spent AS (
    SELECT
        pr.category,
        SUM(p.quantity * pr.price) AS spent_18_25
    FROM purchases p
    JOIN products pr ON p.product_id = pr.product_id
    JOIN users u ON p.user_id = u.user_id
    WHERE u.age BETWEEN 18 AND 25
    GROUP BY pr.category
), total AS (
    SELECT SUM(spent_18_25) AS total_spent FROM category_spent
)
SELECT
    c.category,
    ROUND(c.spent_18_25, 2) AS spent_18_25,
    ROUND((c.spent_18_25 / t.total_spent) * 100, 2) AS share_percent
FROM category_spent c
CROSS JOIN total t
ORDER BY share_percent DESC
""")

share_18_25_by_category.show(truncate=False)

+-----------+-----------+-------------+
|category   |spent_18_25|share_percent|
+-----------+-----------+-------------+
|Home       |361.1      |29.9         |
|Sports     |310.5      |25.71        |
|Electronics|249.6      |20.67        |
|Clothing   |245.0      |20.29        |
|Beauty     |41.4       |3.43         |
+-----------+-----------+-------------+



In [9]:
# 6. Топ-3 категорії за часткою витрат
top_3_categories = spark.sql("""
SELECT *
FROM (
    WITH category_spent AS (
        SELECT
            pr.category,
            SUM(p.quantity * pr.price) AS spent_18_25
        FROM purchases p
        JOIN products pr ON p.product_id = pr.product_id
        JOIN users u ON p.user_id = u.user_id
        WHERE u.age BETWEEN 18 AND 25
        GROUP BY pr.category
    ), total AS (
        SELECT SUM(spent_18_25) AS total_spent FROM category_spent
    )
    SELECT
        c.category,
        ROUND(c.spent_18_25, 2) AS spent_18_25,
        ROUND((c.spent_18_25 / t.total_spent) * 100, 2) AS share_percent
    FROM category_spent c
    CROSS JOIN total t
    ORDER BY share_percent DESC
) ranked
LIMIT 3
""")

top_3_categories.show(truncate=False)

+-----------+-----------+-------------+
|category   |spent_18_25|share_percent|
+-----------+-----------+-------------+
|Home       |361.1      |29.9         |
|Sports     |310.5      |25.71        |
|Electronics|249.6      |20.67        |
+-----------+-----------+-------------+



In [10]:
spark.stop()